In [ ]:
import sys
import logging
from pathlib import Path

import torch

PROJECT_ROOT = Path("../..").resolve()
sys.path.insert(0, str(PROJECT_ROOT))

from utils.io_utils import load_fold
from utils.mlp_utils_lo import (
    set_seed,
    prepare_all_fold_tensors_lo,
    run_nested_random_search_lo,
    print_final_results_lo,
)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)

logger = logging.getLogger(__name__)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
logger.info(f"Device: {device}")

GLOBAL_SEED = 42
set_seed(GLOBAL_SEED)

00:53:15 [INFO] Device: cuda


In [2]:
CFG = {
    "task":        "lo",
    "dataset":     "drd2",
    "fp_type":     "rdkit_desc",
    "n_bits":      1024,
    "outer_folds": [1, 2, 3],
    "inner_k":     2,
    "random_state": GLOBAL_SEED,
}

In [3]:
SEARCH_SPACE = {
    "n_layers":     [1, 2, 3],
    "n_nodes":      [64, 128, 256, 512, 1024],
    "r":            [0.5, 0.7, 1.0],
    "activation":   ["relu", "leaky_relu", "gelu", "silu"],  
    "dropout":      [0.0, 0.2, 0.3, 0.5],
    "batchnorm":    [True, False],
    "init":         ["kaiming", "xavier"],

    "lr":           [1e-4, 5e-4, 1e-3, 2e-3, 3e-3],
    "weight_decay": [0.0, 1e-5, 1e-4, 1e-3, 5e-3, 1e-2],
    "batch_size":   [64, 128, 256],
}

FIXED_HP = {
    "max_epochs": 100,
    "patience":   10,
    "grad_clip":  1.0,
}

N_ITER  = 150
N_SEEDS = 3

In [4]:
folds_data = {}

for fold_idx in CFG["outer_folds"]:
    train_df, test_df = load_fold(CFG["task"], CFG["dataset"], fold_idx)

    folds_data[fold_idx] = {"train": train_df, "test": test_df}

    logger.info(
        f"Fold {fold_idx} | "
        f"train={len(train_df)} "
        f"y_mean={train_df['value'].mean():.3f} "
        f"y_std={train_df['value'].std():.3f} | "
        f"test={len(test_df)} "
        f"n_clusters={test_df['cluster'].nunique()}"
    )

folds_tensors = prepare_all_fold_tensors_lo(
    cfg=CFG,
    folds_data=folds_data,
    logger=logger,
)

00:53:15 [INFO] Loaded lo/drd2 fold 1: train=2206, test=267
00:53:15 [INFO] Fold 1 | train=2206 y_mean=6.721 y_std=0.867 | test=267 n_clusters=50
00:53:15 [INFO] Loaded lo/drd2 fold 2: train=2128, test=267
00:53:15 [INFO] Fold 2 | train=2128 y_mean=6.736 y_std=0.840 | test=267 n_clusters=50
00:53:15 [INFO] Loaded lo/drd2 fold 3: train=2257, test=262
00:53:15 [INFO] Fold 3 | train=2257 y_mean=6.753 y_std=0.857 | test=262 n_clusters=50
00:53:15 [INFO] Loading fingerprints from cache: /home/f.capria/drug-discovery-lohi/features/lo/drd2/rdkit_desc_train_1.npz
00:53:15 [INFO] Loading fingerprints from cache: /home/f.capria/drug-discovery-lohi/features/lo/drd2/rdkit_desc_test_1.npz


00:53:15 [INFO] Fold 1 | X_train: (2206, 217), X_test: (267, 217) | scale_features=True | y_mean=6.721 y_std=0.867
00:53:15 [INFO] Loading fingerprints from cache: /home/f.capria/drug-discovery-lohi/features/lo/drd2/rdkit_desc_train_2.npz
00:53:15 [INFO] Loading fingerprints from cache: /home/f.capria/drug-discovery-lohi/features/lo/drd2/rdkit_desc_test_2.npz
00:53:15 [INFO] Fold 2 | X_train: (2128, 217), X_test: (267, 217) | scale_features=True | y_mean=6.736 y_std=0.840
00:53:15 [INFO] Loading fingerprints from cache: /home/f.capria/drug-discovery-lohi/features/lo/drd2/rdkit_desc_train_3.npz
00:53:15 [INFO] Loading fingerprints from cache: /home/f.capria/drug-discovery-lohi/features/lo/drd2/rdkit_desc_test_3.npz
00:53:15 [INFO] Fold 3 | X_train: (2257, 217), X_test: (262, 217) | scale_features=True | y_mean=6.753 y_std=0.857


In [5]:
logger.info(f"Estimated time: ~{N_ITER * 6 * len(CFG['outer_folds']) / 60:.0f} min")

fold_results = run_nested_random_search_lo(
    cfg=CFG,
    folds_tensors=folds_tensors,
    folds_data=folds_data,
    search_space=SEARCH_SPACE,
    fixed_hp=FIXED_HP,
    n_iter=N_ITER,
    n_seeds=N_SEEDS,
    device=device,
    seed=GLOBAL_SEED,
    logger=logger,
)

print_final_results_lo(fold_results, title="MLP DRD2 Lo — rdkit_desc")

00:53:15 [INFO] Estimated time: ~45 min
00:53:15 [INFO] 
OUTER FOLD 1
00:53:19 [INFO]   [1/150] inner MSE=0.4739 (score=-0.4739) (4s) | L=2 N=512 r=0.7 dropout=0.3 lr=3e-03
00:53:25 [INFO]   [2/150] inner MSE=0.5610 (score=-0.5610) (5s) | L=3 N=128 r=1.0 dropout=0.2 lr=1e-03
00:53:27 [INFO]   [3/150] inner MSE=0.5866 (score=-0.5866) (2s) | L=1 N=1024 r=0.7 dropout=0.0 lr=5e-04
00:53:31 [INFO]   [4/150] inner MSE=0.5332 (score=-0.5332) (5s) | L=2 N=64 r=0.7 dropout=0.3 lr=3e-03
00:53:33 [INFO]   [5/150] inner MSE=0.5494 (score=-0.5494) (2s) | L=2 N=256 r=1.0 dropout=0.5 lr=3e-03
00:53:37 [INFO]   [6/150] inner MSE=0.7344 (score=-0.7344) (4s) | L=1 N=64 r=0.7 dropout=0.5 lr=1e-04
00:53:43 [INFO]   [7/150] inner MSE=0.7532 (score=-0.7532) (6s) | L=3 N=128 r=1.0 dropout=0.3 lr=1e-03
00:53:48 [INFO]   [8/150] inner MSE=1.8776 (score=-1.8776) (5s) | L=2 N=64 r=0.7 dropout=0.5 lr=5e-04
00:53:51 [INFO]   [9/150] inner MSE=365.1653 (score=-365.1653) (3s) | L=1 N=256 r=0.5 dropout=0.3 lr=1e-04
0


MLP DRD2 Lo — rdkit_desc
Fold 1: mean_spearman=0.2485
Fold 2: mean_spearman=0.3083
Fold 3: mean_spearman=0.2628

Aggregated metrics:
  mean_spearman_mean: 0.2732
  mean_spearman_std: 0.0255
  std_spearman_mean: 0.4491
  std_spearman_std: 0.0295
  mean_r2_mean: -0.6708
  mean_r2_std: 0.0738
  mean_mae_mean: 0.7958
  mean_mae_std: 0.0161
  n_clusters_mean: 50.0
  n_clusters_std: 0.0

Best hyperparameters per fold:
Fold 1: L=2 N=256 r=0.7 act=gelu dropout=0.2 bn=True init=kaiming lr=1e-03 wd=5e-03 bs=256
Fold 2: L=2 N=256 r=0.7 act=relu dropout=0.3 bn=True init=xavier lr=1e-03 wd=1e-04 bs=64
Fold 3: L=2 N=1024 r=1.0 act=relu dropout=0.2 bn=False init=kaiming lr=2e-03 wd=1e-02 bs=128


{'mean_spearman_mean': np.float64(0.2732),
 'mean_spearman_std': np.float64(0.0255),
 'std_spearman_mean': np.float64(0.4491),
 'std_spearman_std': np.float64(0.0295),
 'mean_r2_mean': np.float64(-0.6708),
 'mean_r2_std': np.float64(0.0738),
 'mean_mae_mean': np.float64(0.7958),
 'mean_mae_std': np.float64(0.0161),
 'n_clusters_mean': np.float64(50.0),
 'n_clusters_std': np.float64(0.0)}

In [6]:
import json
from pathlib import Path

out_dir = PROJECT_ROOT / "results" / CFG["task"] / CFG["dataset"] / f"mlp_{CFG['dataset']}_{CFG['task']}_{CFG['fp_type']}"
out_dir.mkdir(parents=True, exist_ok=True)

for res in fold_results:
    fold = res["fold"]

    with open(out_dir / f"params_fold_{fold}.json", "w") as f:
        json.dump({
            "fold": fold,
            "best_params": res["best_hp"],
            "inner_cv_score": res["inner_score"],
            "inner_train_loss_diagnostic": res["best_train_loss_diagnostic"],
            "test_metrics": res["test_metrics"],
            "seed_results": res["seed_results"],
        }, f, indent=2, default=str)

print(f"Saved JSON files in: {out_dir}")

Saved JSON files in: /home/f.capria/drug-discovery-lohi/results/lo/drd2/mlp_drd2_lo_rdkit_desc
